# Differential Geometry for Geophysical Fluid Dynamics

Symbolic differential geometry with SageManifolds, following the tensor
formulation of

> S. Gaudreault (ECCC, Dorval), S. Taylor and R. J. Spiteri,
> *Geometric Approach to Geophysical Fluid Dynamics*


The paper augments Newtonian spacetime with a fifth, null coordinate $x^4$ and
writes the Eisenhart–Duval line element

$$ d\ell^2 = G_{AB}\,dX^A dX^B = G_{ij}\,dx^i dx^j + 2G_{0i}\,dx^i dx^0 - 2\Phi_e\,(dx^0)^2 + 2\,dx^0 dx^4 , $$

with $\Phi_e$ the effective gravitational potential and $G_{0i}$ carrying the
motion of the frame, rotation included. 

The point of doing this in Sage is that the connection coefficients, covariant
divergences and curvature are then computed rather than derived by hand.

Index conventions, matching the paper: $A,B,C \in \{0,\dots,4\}$;
$\mu,\nu \in \{0,\dots,3\}$; $i,j,k \in \{1,2,3\}$. Sage defaults to
`start_index=0`, so pass `start_index=1` when building a spatial 3-manifold if
you want its indices to read as the paper's $i,j,k$.



### Package imports

In [1]:

# SageManifolds is sage.manifolds, part of SageMath since 7.5. Preloaded on the
# SageMath kernel; this import also covers a plain Python kernel.
from sage.all import *
from sage.manifolds.manifold import Manifold
from sage.manifolds.chart import Chart
from sage.calculus.var import var, function

# %display latex is a magic from Sage's IPython extension, absent on a plain
# Python kernel.
from IPython import get_ipython
_ip = get_ipython()
if _ip is not None and 'display' in _ip.magics_manager.magics['line']:
    _ip.run_line_magic('display', 'latex')
else:
    from sage.repl.rich_output import get_display_manager
    get_display_manager().preferences.text = 'latex'

from sage.version import version
print(version)
Manifold.options.omit_function_arguments = True

10.9


## Manifold and metric declaration

In [2]:
M = Manifold(5, 'M', structure='Lorentzian', metric_name='G')

# Coordinates X^A = (x0, ..., x4), x0 the time and x4 the auxiliary fibre
# coordinate. Text after each colon is the LaTeX symbol; none of the five is
# range-restricted.
X = M.chart(r'x0:x^0 x1:x^1 x2:x^2 x3:x^3 x4:x^4')
x0, x1, x2, x3, x4 = X[:]
show(X)

# structure='Lorentzian' already created the metric under metric_name, so
# retrieve it rather than declaring a second one.
G = M.metric()

# Every component is an unspecified function of time and the three spatial
# coordinates, but not of x4
coords = (x0, x1, x2, x3)

# Effective potential
Phi_e = function('Phi_e', latex_name=r'\Phi_e')(*coords)

# Frame motion, rotation included. Defined as a dictionary of its second index
G0 = {i: function('G0%d' % i, latex_name=r'G_{0%d}' % i)(*coords)
      for i in (1, 2, 3)}

# Positive-definite spatial 3-metric. Only the upper triangle is built.
Gij = {(i, j): function('G%d%d' % (i, j), latex_name=r'G_{%d%d}' % (i, j))(*coords)
       for i in (1, 2, 3) for j in (1, 2, 3) if i <= j}

# Eq (16). G[i,4] and G[4,4] are left unset, hence zero: the fibre direction is
# null and orthogonal to the spatial block.
G[0, 0] = -2*Phi_e
G[0, 4] = 1
for i, f in G0.items(): # so G_0i=f
    G[0, i] = f
for (i, j), f in Gij.items():
    G[i, j] = f

show(G.display())
show(G[:])


Chart (M, (x0, x1, x2, x3, x4))
G = -2*Phi_e dx0⊗dx0 + G01 dx0⊗dx1 + G02 dx0⊗dx2 + G03 dx0⊗dx3 + dx0⊗dx4 + G01 dx1⊗dx0 + G11 dx1⊗dx1 + G12 dx1⊗dx2 + G13 dx1⊗dx3 + G02 dx2⊗dx0 + G12 dx2⊗dx1 + G22 dx2⊗dx2 + G23 dx2⊗dx3 + G03 dx3⊗dx0 + G13 dx3⊗dx1 + G23 dx3⊗dx2 + G33 dx3⊗dx3 + dx4⊗dx0
[-2*Phi_e      G01      G02      G03        1]
[     G01      G11      G12      G13        0]
[     G02      G12      G22      G23        0]
[     G03      G13      G23      G33        0]
[       1        0        0        0        0]


## Finding the inverse metric

In [3]:
# Inverse metric obtained by inverting the component matrix.
Ginv = G.inverse()
# If you want the matrix form of G^{AB}, uncomment the following. It has very long expressions because it is not yet simplified.
#show(Ginv[:])

##### Confirm that the spatial part is indeed the inverse of $G_{ij}$.

In [4]:
# Method 1: matrix product

# Spatial blocks of G_AB and of G^AB as 3x3 symbolic matrices. 
# .expr() turns each chart function into a symbolic expression.
G_spatial = matrix(SR, 3, 3, lambda a, b: G[a+1, b+1].expr())
Ginv_spatial = matrix(SR, 3, 3, lambda a, b: Ginv[a+1, b+1].expr())

# Eq (18): G_ij G^jk = delta_i^k. The raw product is a mess of unexpanded
# cofactors, so simplify before comparing.
product_1 = (G_spatial * Ginv_spatial).simplify_full()
show(product_1)
print('equals the identity:', product_1 == identity_matrix(SR, 3))

[1 0 0]
[0 1 0]
[0 0 1]
equals the identity: True


### The spatial manifold

The above method does not use the full power of index gymnastics in Sage.

The same check by Einstein summation: Index-notation contraction sums over the
manifold's whole index range, so on $M$ it runs over $A = 0,\dots,4$; however, spatial indices $i,j,k$ run over $1,2,3$. The spatial block needs its own 3-manifold.


In [5]:
# Method 2: Einstein summation


# The spatial block as a 3-manifold in its own right. start_index=1 makes the
# index strings range over the paper's i,j,k = 1,2,3; x0 appears in the
# component functions as a parameter rather than a coordinate.
S = Manifold(3, 'S', latex_name=r'\Sigma', structure='Riemannian', start_index=1)
XS = S.chart('x1 x2 x3')

# G_ij, the spatial block of the metric.
h = S.metric('h')
for (i, j), f in Gij.items():
    h[i, j] = f

# Define G_{0i} as a vector on S.
G0i = S.tensor_field(0, 1, name='G0i', latex_name=r'G_{0i}')
for i in (1, 2, 3):           
    G0i[i] = G[0, i].expr()

# G^ij, the spatial block of the 5x5 inverse, carried onto S as a rank-2
# contravariant tensor. sym=(0,1) declares symmetry in
# the two indices, so assigning the upper triangle fills the rest.
GS = S.tensor_field(2, 0, name='GS', latex_name=r'G^{ij}', sym=(0,1))
for i in (1, 2, 3):
    for j in (1, 2, 3):
        if j >= i:
            GS[i, j] = Ginv[i, j].expr()

# Define G^{4i} as a covector on S.
Ginv4i = S.tensor_field(1, 0, name='Ginv4i', latex_name=r'G^{4i}')
for i in (1, 2, 3):           
    Ginv4i[i] = Ginv[4, i].expr()


##### Confirm that $G^{ij}G_{jk}=\delta^{i}_k$.

In [6]:
# The contraction now runs over k = 1,2,3 only, and returns the
# identity, so G^ij really is the inverse of the spatial block:
product_2 = h['_ij'] * GS['^jk']
show(product_2.display())
show(product_2[:])

∂/∂x1⊗dx1 + ∂/∂x2⊗dx2 + ∂/∂x3⊗dx3
[1 0 0]
[0 1 0]
[0 0 1]


##### Confirm that $G^{4i}=-G^{ij}G_{j0}$.

In [7]:
product_3 = Ginv4i + GS['^ij']* G0i['_j']
show(product_3)
show(product_3[:])

Vector field on the 3-dimensional Riemannian manifold S
[0, 0, 0]


##### Confirm that $G^{0A}=\delta^A_4$.

In [8]:
show(Ginv[:][0])

(0, 0, 0, 0, 1)


##### Confirm that $G^{44}=G_{0i}G^{ij}G_{0j}+2\Phi$.

In [9]:
# Note: G_{0i}G^{ij}G_{0j} lives on the spatial manifold S, while Ginv[4, 4] lives on M.
# Therefore, you should take .expr() before subtracting them.

scal = ((G0i['_i'] * GS['^ij'])['^j'] * G0i['_j']).expr()
show((Ginv[4, 4].expr() - (scal + 2. * Phi_e)).simplify_full())

0


### Some Christoffel symbols

#### Verify that $\Gamma^0_{AB}=0$ for any synchronous coordinate system.

In [10]:
Gamma = G.christoffel_symbols()
# Gamma^0_AB = 0 for all A, B.
print('Gamma^0_AB all zero:',
      all(Gamma[0,A,B] == 0 for A in range(5) for B in range(5)))

Gamma^0_AB all zero: True


#### Verify that $\Gamma^A_{4B}=0$.

In [11]:
print('Gamma^A_4B all zero:',
      all(Gamma[A,4,B] == 0 for A in range(5) for B in range(5)))

Gamma^A_4B all zero: True


In [12]:
# Every non-vanishing connection coefficient.
show(G.christoffel_symbols_display())

Gam^x1_x0,x0 = ((G23^2 - G22*G33)*d(G01)/dx0 - (G13*G23 - G12*G33)*d(G02)/dx0 + (G13*G22 - G12*G23)*d(G03)/dx0 + (G23^2 - G22*G33)*d(Phi_e)/dx1 - (G13*G23 - G12*G33)*d(Phi_e)/dx2 + (G13*G22 - G12*G23)*d(Phi_e)/dx3)/(G13^2*G22 - 2*G12*G13*G23 + G11*G23^2 + (G12^2 - G11*G22)*G33) 
Gam^x1_x0,x1 = 1/2*((G13*G23 - G12*G33)*d(G01)/dx2 - (G13*G22 - G12*G23)*d(G01)/dx3 - (G13*G23 - G12*G33)*d(G02)/dx1 + (G13*G22 - G12*G23)*d(G03)/dx1 + (G23^2 - G22*G33)*d(G11)/dx0 - (G13*G23 - G12*G33)*d(G12)/dx0 + (G13*G22 - G12*G23)*d(G13)/dx0)/(G13^2*G22 - 2*G12*G13*G23 + G11*G23^2 + (G12^2 - G11*G22)*G33) 
Gam^x1_x0,x2 = 1/2*((G23^2 - G22*G33)*d(G01)/dx2 - (G23^2 - G22*G33)*d(G02)/dx1 - (G13*G22 - G12*G23)*d(G02)/dx3 + (G13*G22 - G12*G23)*d(G03)/dx2 + (G23^2 - G22*G33)*d(G12)/dx0 - (G13*G23 - G12*G33)*d(G22)/dx0 + (G13*G22 - G12*G23)*d(G23)/dx0)/(G13^2*G22 - 2*G12*G13*G23 + G11*G23^2 + (G12^2 - G11*G22)*G33) 
Gam^x1_x0,x3 = 1/2*((G23^2 - G22*G33)*d(G01)/dx3 - (G13*G23 - G12*G33)*d(G02)/dx3 - (G23^2 - G22*G

### 3.c Example: particle in a gravitational potential

Specialize eq (16) to a static potential $\Phi_e = \Phi$, no rotation
($G_{0i} = 0$), and Cartesian spatial coordinates, giving eq (22):

$$ d\ell^2 = \delta_{ij}\,dx^i dx^j - 2\Phi\,dt^2 + 2\,dt\,dx^4 . $$


In [13]:
# A second manifold, so the general metric above stays untouched. Coordinates
# (t, x, y, z, w): t the time, w the fibre, LaTeX symbols matching the paper's
# X^A = (t, x^1, x^2, x^3, x^4).
Mc = Manifold(5, 'Mc', latex_name=r'M_{\rm cart}',
              structure='Lorentzian', metric_name='G')
Xc = Mc.chart(r't x:x^1 y:x^2 z:x^3 w:x^4')
t, x, y, z, w = Xc[:]

# Static potential: no t dependence, hence no G_00,0 term below.
Phi = function('Phi', latex_name=r'\Phi')(x, y, z)

# Eq (22). G_0i = 0 and G_ij = delta_ij; G_i4 and G_44 stay unset, i.e. zero.
Gc = Mc.metric()
Gc[0,0] = -2*Phi
Gc[0,4] = 1
Gc[1,1] = 1
Gc[2,2] = 1
Gc[3,3] = 1
show(Gc.display())

# Every non-vanishing connection coefficient.
show(Gc.christoffel_symbols_display())

Gam = Gc.christoffel_symbols()

# Gamma^0_AB = 0 for all A, B: the claim that makes t an affine parameter.
print('Gamma^0_AB all zero:',
      all(Gam[0,A,B] == 0 for A in range(5) for B in range(5)))

# Eq (24): Gamma^i_00 = d(Phi)/dx^i.
for i, xi in zip((1,2,3), (x,y,z)):
    print('Gamma^%d_00 - dPhi/dx^%d =' % (i,i),
          (Gam[i,0,0].expr() - diff(Phi, xi)).simplify_full())

# Eq (25) follows if Gamma^i_BC vanishes except for B = C = 0, since then the
# geodesic equation retains only the u^0 u^0 term.
print('other Gamma^i_BC all zero:',
      all(Gam[i,A,B] == 0 for i in (1,2,3) for A in range(5)
          for B in range(5) if (A,B) != (0,0)))


G = -2*Phi dt⊗dt + dt⊗dw + dx⊗dx + dy⊗dy + dz⊗dz + dw⊗dt
Gam^x_tt = d(Phi)/dx 
Gam^y_tt = d(Phi)/dy 
Gam^z_tt = d(Phi)/dz 
Gam^w_tx = -d(Phi)/dx 
Gam^w_ty = -d(Phi)/dy 
Gam^w_tz = -d(Phi)/dz 
Gamma^0_AB all zero: True
Gamma^1_00 - dPhi/dx^1 = 0
Gamma^2_00 - dPhi/dx^2 = 0
Gamma^3_00 - dPhi/dx^3 = 0
other Gamma^i_BC all zero: True


## Spherical coordinates

In [14]:
# A third manifold, so the general metric above stays untouched. Coordinates
# X^A = (t, l, phi, r, x4): t the time, l the longitude in the rotating frame,
# phi the geographic latitude, r the distance from the planet's centre, x4 the
# fibre. The ranges keep sqrt|det G| free of absolute values.
M_spherical = Manifold(5, 'M_spherical', latex_name=r'M_{\rm spherical}',
              structure='Lorentzian', metric_name='G_s')
Xs = M_spherical.chart(r't l:\lambda phi:(-pi/2,pi/2):\varphi r:(0,+oo) x4:x^4')
t, l, phi, r, x4 = Xs[:]

# declare constants 

Omega = var('Omega', latex_name=r'\Omega', domain='real')   # earth rotation frequency
G_N = var('G_N', latex_name='G_N', domain='positive')       # Newton's gravitational constant    
M_E = var('M_E', latex_name='M_E', domain='positive')       # Mass of Earth


# a rotating spherical metric in terms of latitude phi and longitude l
G_s = M_spherical.metric()
G_s[0,0] = -2*(-M_E*G_N/r-  Omega**2/2 * r**2 * cos(phi)**2)
G_s[0,4] = 1
G_s[0,1] = Omega * r**2 * cos(phi)**2
G_s[1,1] = r**2 * cos(phi)**2
G_s[2,2] = r**2
G_s[3,3] = 1
show(G_s.display())
show(G_s[:])

Gamma_s = G_s.christoffel_symbols()

G_s = (Omega^2*r^2*cos(phi)^2 + 2*G_N*M_E/r) dt⊗dt + Omega*r^2*cos(phi)^2 dt⊗dl + dt⊗dx4 + Omega*r^2*cos(phi)^2 dl⊗dt + r^2*cos(phi)^2 dl⊗dl + r^2 dphi⊗dphi + dr⊗dr + dx4⊗dt
[Omega^2*r^2*cos(phi)^2 + 2*G_N*M_E/r                 Omega*r^2*cos(phi)^2                                    0                                    0                                    1]
[                Omega*r^2*cos(phi)^2                       r^2*cos(phi)^2                                    0                                    0                                    0]
[                                   0                                    0                                  r^2                                    0                                    0]
[                                   0                                    0                                    0                                    1                                    0]
[                                   1                                    0    

#### The inverse metric

In [15]:
Gsinv = G_s.inverse()
show(Gsinv[:])

[                 0                  0                  0                  0                  1]
[                 0 1/(r^2*cos(phi)^2)                  0                  0             -Omega]
[                 0                  0             r^(-2)                  0                  0]
[                 0                  0                  0                  1                  0]
[                 1             -Omega                  0                  0       -2*G_N*M_E/r]


#### Verify that $\Gamma^0_{AB}=0$:

In [16]:
# Gamma^0_AB = 0 for all A, B.
print('Gamma^0_AB all zero:',
      all(Gamma_s[0,A,B] == 0 for A in range(5) for B in range(5)))

Gamma^0_AB all zero: True


#### $\Gamma^{1}_{AB}$ as a matrix:

In [17]:
Gamma1_AB = matrix(SR, 5, 5, lambda a, b: Gamma_s[1,a, b].expr()).simplify_full()
show(Gamma1_AB)

[                       0                        0 -Omega*sin(phi)/cos(phi)                  Omega/r                        0]
[                       0                        0       -sin(phi)/cos(phi)                      1/r                        0]
[-Omega*sin(phi)/cos(phi)       -sin(phi)/cos(phi)                        0                        0                        0]
[                 Omega/r                      1/r                        0                        0                        0]
[                       0                        0                        0                        0                        0]


#### $\Gamma^{2}_{AB}$ as a matrix:

In [18]:
Gamma2_AB = matrix(SR, 5, 5, lambda a, b: Gamma_s[2,a, b].expr()).simplify_full()
show(Gamma2_AB)

[Omega^2*cos(phi)*sin(phi)   Omega*cos(phi)*sin(phi)                         0                         0                         0]
[  Omega*cos(phi)*sin(phi)         cos(phi)*sin(phi)                         0                         0                         0]
[                        0                         0                         0                       1/r                         0]
[                        0                         0                       1/r                         0                         0]
[                        0                         0                         0                         0                         0]


#### $\Gamma^{3}_{AB}$ as a matrix:

In [19]:
Gamma3_AB = matrix(SR, 5, 5, lambda a, b: Gamma_s[3,a, b].expr()).simplify_full()
show(Gamma3_AB)

[-(Omega^2*r^3*cos(phi)^2 - G_N*M_E)/r^2                     -Omega*r*cos(phi)^2                                       0                                       0                                       0]
[                    -Omega*r*cos(phi)^2                           -r*cos(phi)^2                                       0                                       0                                       0]
[                                      0                                       0                                      -r                                       0                                       0]
[                                      0                                       0                                       0                                       0                                       0]
[                                      0                                       0                                       0                                       0                                

#### $\Gamma^{4}_{AB}$ as a matrix:

In [20]:
Gamma4_AB = matrix(SR, 5, 5, lambda a, b: Gamma_s[4,a, b].expr()).simplify_full()
show(Gamma4_AB)

[           0            0            0 -G_N*M_E/r^2            0]
[           0            0            0            0            0]
[           0            0            0            0            0]
[-G_N*M_E/r^2            0            0            0            0]
[           0            0            0            0            0]


## Thin shell approximation

In [21]:
# Coordinates X^A = (t, l, phi, r, x4): t the time, 
# l the longitude in the rotating frame,
# phi the geographic latitude, z the distance from a reference radius a, x4 the
# fibre. The ranges keep sqrt|det G| free of absolute values.
M_shell = Manifold(5, 'M_shell', latex_name=r'M_{\rm shell}',
              structure='Lorentzian', metric_name='G_shell')
Xshell = M_shell.chart(r't l:\lambda phi:(-pi/2,pi/2):\varphi z:x^3 x4:x^4')
t_shell, l_shell, phi_shell, z, x4_shell = Xshell[:]

# declare constants 

Omega = var('Omega', latex_name=r'\Omega', domain='real')   # earth rotation frequency
g = var('g', latex_name='g', domain='positive')             # Earth's gravitational acceleration   
a = var('a', latex_name='a', domain='positive')             # reference radius


# a rotating spherical metric in terms of latitude phi and longitude l
G_shell = M_shell.metric()
G_shell[0,0] = -2*g*z
G_shell[0,4] = 1
G_shell[0,1] = Omega * a**2 * cos(phi_shell)**2
G_shell[1,1] = a**2 * cos(phi_shell)**2
G_shell[2,2] = a**2
G_shell[3,3] = 1
show(G_shell.display())
show(G_shell[:])

Gamma_shell = G_shell.christoffel_symbols()

G_shell = -2*g*z dt⊗dt + Omega*a^2*cos(phi)^2 dt⊗dl + dt⊗dx4 + Omega*a^2*cos(phi)^2 dl⊗dt + a^2*cos(phi)^2 dl⊗dl + a^2 dphi⊗dphi + dz⊗dz + dx4⊗dt
[              -2*g*z Omega*a^2*cos(phi)^2                    0                    0                    1]
[Omega*a^2*cos(phi)^2       a^2*cos(phi)^2                    0                    0                    0]
[                   0                    0                  a^2                    0                    0]
[                   0                    0                    0                    1                    0]
[                   1                    0                    0                    0                    0]


#### Verify that $\Gamma^0_{AB}=0$:

In [23]:
# Gamma^0_AB = 0 for all A, B.
print('Gamma^0_AB all zero:',
      all(Gamma_shell[0,A,B] == 0 for A in range(5) for B in range(5)))

Gamma^0_AB all zero: True


#### $\Gamma^{1}_{AB}$ as a matrix:

In [ ]:
Gamma1_AB = matrix(SR, 5, 5, lambda a, b: Gamma_shell[1,a, b].expr()).simplify_full()
show(Gamma1_AB)

[                       0                        0 -Omega*sin(phi)/cos(phi)                        0                        0]
[                       0                        0       -sin(phi)/cos(phi)                        0                        0]
[-Omega*sin(phi)/cos(phi)       -sin(phi)/cos(phi)                        0                        0                        0]
[                       0                        0                        0                        0                        0]
[                       0                        0                        0                        0                        0]


#### $\Gamma^{2}_{AB}$ as a matrix:

In [25]:
Gamma2_AB = matrix(SR, 5, 5, lambda a, b: Gamma_shell[2,a, b].expr()).simplify_full()
show(Gamma2_AB)

[                      0 Omega*cos(phi)*sin(phi)                       0                       0                       0]
[Omega*cos(phi)*sin(phi)       cos(phi)*sin(phi)                       0                       0                       0]
[                      0                       0                       0                       0                       0]
[                      0                       0                       0                       0                       0]
[                      0                       0                       0                       0                       0]


#### $\Gamma^{3}_{AB}$ as a matrix:

In [26]:
Gamma3_AB = matrix(SR, 5, 5, lambda a, b: Gamma_shell[3,a, b].expr()).simplify_full()
show(Gamma3_AB)

[g 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]
[0 0 0 0 0]


#### $\Gamma^{4}_{AB}$ as a matrix:

In [27]:
Gamma4_AB = matrix(SR, 5, 5, lambda a, b: Gamma_shell[4,a, b].expr()).simplify_full()
show(Gamma4_AB)

[                            0                             0 Omega^2*a^2*cos(phi)*sin(phi)                            -g                             0]
[                            0                             0                             0                             0                             0]
[Omega^2*a^2*cos(phi)*sin(phi)                             0                             0                             0                             0]
[                           -g                             0                             0                             0                             0]
[                            0                             0                             0                             0                             0]
